# GPU en Colab: benchmark ASR + transcripción del corpus

Este cuaderno es lo que corre **una vez**. Todo lo que produce (`cache/`, `results/`) se descarga y se versiona;
la app, la evaluación y el CI solo leen esa caché.

1. Entorno de ejecución → Cambiar tipo → **GPU T4**.
2. Sube tus audios a Google Drive en `MyDrive/vozregistro_audio/interview/E01.m4a ...` y `.../order/P01.m4a ...`
   (los nombres deben coincidir con los guiones de `data/scripts/`).

In [ ]:
!nvidia-smi

In [ ]:
REPO = "https://github.com/TU_USUARIO/HW5-DS.git"   # <- cambia esto
!git clone $REPO repo
%cd repo
!pip install -q -r requirements.txt

In [ ]:
# Prueba de que la GPU es visible PARA LA LIBRERÍA (no para la configuración)
import ctranslate2
print("CUDA devices visibles por CTranslate2:", ctranslate2.get_cuda_device_count())
print("compute types cuda:", ctranslate2.get_supported_compute_types("cuda"))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p data/audio && cp -r /content/drive/MyDrive/vozregistro_audio/* data/audio/
!ls -R data/audio

In [ ]:
!python scripts/build_ground_truth.py

## Antes del corpus completo: dos clips de punta a punta (barato descubrir problemas aquí)

In [ ]:
from vozregistro.engine import process_audio
for p, s in [("data/audio/interview/E01.m4a", "interview"), ("data/audio/order/P01.m4a", "order")]:
    r = process_audio(p, schema=s)
    print(p, r["device"], r["compute_type"], r["asr"] and r["asr"]["peak_vram_mb"], r["error"])
    print(r["fields"], r["verified"], r.get("validation"), sep="
")

## Tarea 1 · Fase 2: tamaños, precisiones, CPU vs GPU, glosario

In [ ]:
!python scripts/benchmark_asr.py

## Transcripciones principales + escalera de degradación

In [ ]:
!python scripts/transcribe_corpus.py --task interview
!python scripts/build_ladder.py --task order
!python scripts/transcribe_corpus.py --task order --levels L0 L1 L2 L3 L4

## Ruta LLM sin API key: Ollama + modelo abierto en la misma GPU

Gratis y local: el texto no sale de la sesión. Antes de cargar el LLM se libera Whisper de la VRAM.

In [ ]:
from vozregistro.asr import release_models
release_models()
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:7b-instruct

In [ ]:
# reglas + LLM (Ollama). Las respuestas quedan en cache/llm/ para no recalcularlas nunca más
!python scripts/evaluate.py

In [ ]:
!zip -qr resultados.zip cache results
from google.colab import files
files.download("resultados.zip")   # descomprime en la raíz del repo local y haz commit de cache/ y results/